In [15]:
import os
from dotenv import load_dotenv

# langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

In [2]:
load_dotenv()

True

In [4]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")

ENV VAR LOADED


In [7]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

DATA INGESSTION

In [8]:
loader = TextLoader(DATA_FILE_PATH , encoding="utf-8")

documents = loader.load()

print("DATA LOADED")
print("=== DOCUMENTS ===")
print(documents)

DATA LOADED
=== DOCUMENTS ===
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their da

In [9]:
print(f"Total characters in the document: {len(documents[0].page_content)}")

Total characters in the document: 2598


SPLITTING OUR DATA


In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [14]:
len(chunks)

9

EMBEDDING OUR DATA

In [18]:
embeddings = JinaEmbeddings(jina_key=jina_key, model_name="jina-embeddings-v2-base-en")

print("EMBEDDINGS MODEL READY THE NAME IS: ", embeddings.model_name)

EMBEDDINGS MODEL READY THE NAME IS:  jina-embeddings-v2-base-en


In [19]:
from langchain_classic.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings)

print("CHUNCKS ARE STORED" , vector_store.index.ntotal)

CHUNCKS ARE STORED 9


In [21]:
test_query = "How many sick leaves employes get"

top_matches = vector_store.similarity_search(test_query, k=3)
print(f"Query: {test_query}\n")
for i, match in enumerate(top_matches):
    print(f"Match {i+1}:")
    print(match.page_content)
    print("\n---\n")

Query: How many sick leaves employes get

Match 1:
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

---

Match 2:
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

---

Match 3:
3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emergencies, subject to manager approval.
Per

TOOL

In [38]:
retriver = vector_store.as_retriever(search_kwargs={"k": 3})

def search_hr_policy(question: str)-> str:
    """
    Search the HR policy document for information about leave, work from home,
    probation , notice period, reimbursement, code of conduct, holidays, or exit process.
    """
    matching_chucks = retriver.invoke(question)
    return "\n\n".join(chunks.page_content for chunks in matching_chucks)

DATA RETRIVAL

In [25]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0,
)
llm.model_name

'openai/gpt-oss-120b'

In [26]:
test_response = llm.invoke("Hey is learning Rag hard? reply in one line")

In [27]:
test_response.content

'Learning RAG can be challenging at first, but with hands‑on practice it becomes manageable.'

In [28]:
from langchain.agents import create_agent

In [46]:
hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""

    You are a friendly HR assistant for Acme Corporation.
    Always use the search_hr_policy tool to look up facts before answering.
    if the answer isn't in the search results, say you don't know instead of guessing.
    """
)
print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [52]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the Rag agent and print a nicely formatted answer."""
    print("="*60)
    print(f"Question: {question}")
    print("="*60)
    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("="*60)
    print("\n\n")
    return answer

In [53]:
response = hr_assistant.invoke(
    {
        "messages": [
            {
                "role": "user", 
                "content": "tell me which org you work for "
            }
        ]
    }
)

In [54]:
response

{'messages': [HumanMessage(content='tell me which org you work for ', additional_kwargs={}, response_metadata={}, id='9bafbbbb-56da-4734-b7c1-16c10e32c359'),
  AIMessage(content='I’m the friendly HR assistant here at **Acme\u202fCorporation**. How can I help you today?', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". The assistant is a friendly HR assistant for Acme Corporation. Should answer that I work for Acme Corporation. No need to search policy. It\'s not about policy. So answer directly.'}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 202, 'total_tokens': 285, 'completion_time': 0.175711284, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.050492723, 'prompt_tokens_details': None, 'queue_time': 0.286680123, 'total_time': 0.226204007}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b1dd3e7a63', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'm

In [55]:
print(response["messages"][-1].content)

I’m the friendly HR assistant here at **Acme Corporation**. How can I help you today?
